In [1]:
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer
from nltk.tokenize import word_tokenize
import logging
import math
from tqdm import tqdm
from joblib import Parallel, delayed
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag

In [2]:
# ISO Latin-1 encoding
df = pd.read_csv("Sentiment_Data.csv", encoding='iso-8859-1')

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
# Mapping 5-class sentiment to 3-class sentiment
sentiment_mapping = {
    'Strong_Pos': 'positive',
    'Mild_Pos': 'positive',
    'Neutral': 'neutral',
    'Mild_Neg': 'negative',
    'Strong_Neg': 'negative'
}

# Apply mapping to target column
df['Sentiment_3_Class'] = df['Sentiment'].map(sentiment_mapping)

# Optional: check value counts
print(df['Sentiment_3_Class'].value_counts())


Sentiment_3_Class
positive    297704
neutral      77016
negative     76612
Name: count, dtype: int64


In [5]:
# Drop the original 5-class sentiment column
df.drop(columns=['Sentiment'], inplace=True)

# Rename the 3-class column to 'Sentiment'
df.rename(columns={'Sentiment_3_Class': 'Sentiment'}, inplace=True)

# Optional: check the result
print(df.head())


                                               Tweet Sentiment
0  @_angelica_toy Happy Anniversary!!!....The Day...  positive
1  @McfarlaneGlenda Happy Anniversary!!!....The D...  positive
2  @thevivafrei @JustinTrudeau Happy Anniversary!...  positive
3  @NChartierET Happy Anniversary!!!....The Day t...  positive
4  @tabithapeters05 Happy Anniversary!!!....The D...  positive


In [6]:
df.sample(10)

,Tweet,Sentiment
390084,The European Freedom Convoy is truckin'! https...,positive
366074,A few rays of Light in this convoy of Darkness...,negative
82079,"@AndrewLawton No, his police just beat protest...",negative
46234,"Poilievre stands by Freedom Convoy support, bu...",positive
198164,@PresidenteMCM Freedom Convoy disrupted your l...,positive
397515,Fiery exchange between Conservatives and Liber...,positive
247256,The irony! Trudeau said martial Law is necessa...,positive
289662,'This could cost him his job': Only 16% of Can...,positive
124740,@Madison46108374 @smalls96 @covid_parent @IanD...,positive
450270,"Marie Colvin Convoy for Freedom in Syria,Convo...",positive


In [7]:
df['Sentiment'].value_counts()

Sentiment
positive    297704
neutral      77016
negative     76612
Name: count, dtype: int64

In [8]:
df.duplicated().sum()

45

In [9]:
df = df.drop_duplicates()

In [10]:
df.duplicated().sum()

0

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 451287 entries, 0 to 451331
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Tweet      451286 non-null  object
 1   Sentiment  451287 non-null  object
dtypes: object(2)
memory usage: 10.3+ MB


In [12]:
df['Sentiment'].value_counts()

Sentiment
positive    297665
neutral      77013
negative     76609
Name: count, dtype: int64

In [13]:
df.isnull().sum()

Tweet        1
Sentiment    0
dtype: int64

In [14]:
df.sample()

,Tweet,Sentiment
171563,@janelleybelley3 @SarahRyanYEG It's quite some...,positive


## Before we do any preprocessing , lets just explore our dataset and identify how many impurities we have

In [15]:
import re
import pandas as pd

def check_preprocessing_targets(df, text_col='Tweet', slang_dict=None):
    """
    Checks if original text column contains patterns like emoji, URL, mentions, hashtags, etc.
    Returns a summary count for each pattern.
    """

    url_pattern = re.compile(r"http\S+|www\S+|https\S+")
    mention_pattern = re.compile(r"@\w+")
    hashtag_pattern = re.compile(r"#\w+")
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F"
             u"\U0001F300-\U0001F5FF"
             u"\U0001F680-\U0001F6FF"
             u"\U0001F1E0-\U0001F1FF" "]+", flags=re.UNICODE)
    number_pattern = re.compile(r"\d+")
    date_pattern = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
    duration_pattern = re.compile(r"\b\d+\s?(hours?|days?|weeks?|mins?|minutes?|24/7)\b", flags=re.IGNORECASE)
    rating_pattern = re.compile(r"\b\d{1,2}/10\b")
    underscore_pattern = re.compile(r"_")

    def contains_slang(text):
        if slang_dict is None:
            return False
        return any(word in slang_dict for word in str(text).split())

    summary = {
        'URLs': df[text_col].apply(lambda x: bool(url_pattern.search(str(x)))).sum(),
        'Mentions (@)': df[text_col].apply(lambda x: bool(mention_pattern.search(str(x)))).sum(),
        'Hashtags (#)': df[text_col].apply(lambda x: bool(hashtag_pattern.search(str(x)))).sum(),
        'Emojis': df[text_col].apply(lambda x: bool(emoji_pattern.search(str(x)))).sum(),
        'Numbers': df[text_col].apply(lambda x: bool(number_pattern.search(str(x)))).sum(),
        'Dates': df[text_col].apply(lambda x: bool(date_pattern.search(str(x)))).sum(),
        'Durations': df[text_col].apply(lambda x: bool(duration_pattern.search(str(x)))).sum(),
        'Ratings (/10)': df[text_col].apply(lambda x: bool(rating_pattern.search(str(x)))).sum(),
        'Underscores': df[text_col].apply(lambda x: bool(underscore_pattern.search(str(x)))).sum(),
        'Slang': df[text_col].apply(contains_slang).sum() if slang_dict else 'Skipped',
    }

    return pd.DataFrame.from_dict(summary, orient='index', columns=['Count']).sort_values(by='Count', ascending=False)


In [16]:
slang_dict = {
    "u": "you",
    "ur": "your",
    "r": "are",
    "wht":'what',
    "lol": "laughing out loud",
    "lmao": "laughing my ass off",
    "rofl": "rolling on the floor laughing",
    "omg": "oh my god",
    "idk": "i don't know",
    "btw": "by the way",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "brb": "be right back",
    "ttyl": "talk to you later",
    "smh": "shaking my head",
    "tbh": "to be honest",
    "dm": "direct message",
    "irl": "in real life",
    "jk": "just kidding",
    "afk": "away from keyboard",
    "np": "no problem",
    "thx": "thanks",
    "ty": "thank you",
    "yw": "you’re welcome",
    "bff": "best friends forever",
    "bday": "birthday",
    "gr8": "great",
    "plz": "please",
    "wtf": "what the fuck",
    "wth": "what the hell",
    "bcoz": "because",
    "coz": "because",
    "cya": "see you",
    "msg": "message",
    "fyi": "for your information",
    "asap": "as soon as possible",
    "atm": "at the moment",
    "bbl": "be back later",
    "cu": "see you",
    "gg": "good game",
    "gl": "good luck",
    "hf": "have fun",
    "hmu": "hit me up",
    "ikr": "i know right"
}

In [17]:
check_preprocessing_targets(df, text_col='Tweet', slang_dict=slang_dict)


,Count
Numbers,280647
URLs,224196
Mentions (@),211489
Hashtags (#),139493
Underscores,32295
Durations,4488
Slang,3164
Dates,973
Ratings (/10),47
Emojis,0


In [18]:
# Step 1: Split first
X = df['Tweet']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [19]:
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, stratify=y_test, test_size=0.5, random_state=42)

In [20]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(361029,)
(45129,)
(361029,)
(45129,)
(45129,)
(45129,)


In [21]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(361029,)
(45129,)
(361029,)
(45129,)
(45129,)
(45129,)


In [22]:
# Step 2: Combine X_train and y_train for sampling
train_df = pd.DataFrame({'Tweet': X_train, 'Sentiment': y_train})

In [23]:
train_df.value_counts('Sentiment')

Sentiment
positive    238132
neutral      61610
negative     61287
Name: count, dtype: int64

In [24]:

# Sample 30000 examples from validation set
val_sample_idx = X_val.sample(n=30000, random_state=40).index
X_val = X_val.loc[val_sample_idx].reset_index(drop=True)
y_val = y_val.loc[val_sample_idx].reset_index(drop=True)

# Sample 20000 examples from test set
test_sample_idx = X_test.sample(n=20000, random_state=50).index
X_test = X_test.loc[test_sample_idx].reset_index(drop=True)
y_test = y_test.loc[test_sample_idx].reset_index(drop=True)

In [25]:
# Step 2: Combine X_train and y_train for sampling
train_df = pd.DataFrame({'Tweet': X_train, 'Sentiment': y_train})

In [26]:
y_train.value_counts()

Sentiment
positive    238132
neutral      61610
negative     61287
Name: count, dtype: int64

In [27]:
from sklearn.utils import resample

# Assuming your DataFrame is called `df`
min_class_size = 61287

# Undersample each class to match the smallest class
train_df = (
    train_df.groupby('Sentiment')
      .apply(lambda x: x.sample(n=min_class_size, random_state=42))
      .reset_index(drop=True)
)

C:\Users\harsh\AppData\Local\Temp\ipykernel_24232\2468175336.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_class_size, random_state=42))


In [28]:
y_val.value_counts()

Sentiment
positive    19768
neutral      5138
negative     5094
Name: count, dtype: int64

In [29]:
X_val.sample()

25012    @Kolmen1989 @QanonAnonymous I hope the "Freedo...
Name: Tweet, dtype: object

In [30]:
val_df = pd.DataFrame({'Tweet': X_val, 'Sentiment': y_val})
val_df = (
    val_df.groupby("Sentiment", group_keys=False)
          .apply(lambda x: x.sample(n=5000, random_state=42))
          .reset_index(drop=True)
)
X_val = val_df["Tweet"]
y_val = val_df["Sentiment"]

C:\Users\harsh\AppData\Local\Temp\ipykernel_24232\1373242052.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=5000, random_state=42))


In [31]:
y_val.value_counts()

Sentiment
negative    5000
neutral     5000
positive    5000
Name: count, dtype: int64

In [32]:
train_df.value_counts('Sentiment')

Sentiment
negative    61287
neutral     61287
positive    61287
Name: count, dtype: int64

In [33]:
# Step 4: Separate again
X_train= train_df['Tweet']
y_train= train_df['Sentiment']

In [34]:
# to comapre results after preprocessing
X_train_backup = X_train.copy()
y_train_backup = y_train.copy()

In [35]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
X_val = pd.DataFrame(X_val)

In [36]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)


(183861, 1)
(183861,)
(20000, 1)
(20000,)
(15000, 1)
(15000,)


## Basic PreProcessing

In [37]:
X_train.head()

,Tweet
0,Ottawa Police Chief swears there will be no re...
1,Paris police ban French âFreedom Convoyâ d...
2,@CandiceBergenMP Are you for the mandates or n...
3,âFreedom Convoyâ protest: Ottawa police pr...
4,#FreedomConvoy Preemptive SOS Press Conference...


In [38]:
def clean_tweet(tweet):
    tweet = str(tweet).lower()
    tweet = re.sub(r"@\S+", '', tweet)  # remove mentions
    tweet = re.sub(r"http\S+|www\S+|https\S+", '', tweet)  # remove URLs
    tweet = re.sub(r"#", '', tweet)  # remove hashtag #
    tweet = re.sub(r"[^\w\s']", ' ', tweet)  # keep words, spaces, and apostrophes
    tweet = re.sub(r'\s+', ' ', tweet).strip()  # remove extra spaces
    return tweet


## Numbers Handling

In [39]:
def smart_number_handler(text):

    # Replace emotional ratings like "0/10" or "10/10"
    text = re.sub(r'\b\d{1,2}/10\b', ' <rating> ', text)

    # Replace durations like "3 hours", "2 days", "24/7"
    text = re.sub(r'\b\d+\s?(hours?|days?|weeks?|mins?|minutes?|24/7)\b', ' <duration> ', text, flags=re.IGNORECASE)

    # Replace dates like "12/03/2022" or "1-1-21"
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', ' <date> ', text)

    # Replace percentages like "90%"
    text = re.sub(r'\b\d+%|\d+\spercent\b', ' <percent> ', text, flags=re.IGNORECASE)

    # Replace currency values like "$200", "€30", "£50.00"
    text = re.sub(r'[$€£]\s?\d+(\.\d+)?', ' <money> ', text)

    # Remove numbers expect for years 1900 - 2100
    text = re.sub(r'\b(?!19\d{2}\b|20\d{2}\b|2100\b)\d+\b', '', text)

    return text


## Emoji -> Text emotions (like happy_smiley_face)

In [40]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.


In [41]:
import emoji

def convert_emojis_to_text(text):
    return emoji.demojize(text, language='en')

def simplify_emoji_text(text):
    text = convert_emojis_to_text(text)
    text = text.replace(":", "")  # remove colons
    text = text.replace("_", " ")  # optional: make it tokenizer friendly
    return text


## Slang and Abbrevation *Dictionary*

## Slang replaceing

In [42]:
def replace_slangs(text):
    words = text.split()
    return ' '.join([slang_dict[word] if word in slang_dict else word for word in words])

## Spell Check

In [43]:
from symspellpy import SymSpell, Verbosity
import importlib.resources

# Initialize SymSpell only once (outside the function)
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = importlib.resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
sym_spell.load_dictionary(str(dictionary_path), term_index=0, count_index=1)

def correct_text_spelling(text):
    try:
        words = text.split()
        corrected_words = []
        for word in words:
            if "'" in word:  # skip contractions like "don't", "I'm"
                corrected_words.append(word)
            else:
                suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
                corrected = suggestions[0].term if suggestions else word
                corrected_words.append(corrected)
        return ' '.join(corrected_words)
    except Exception:
        return text


In [44]:
correct_text_spelling("i don't dont like tis weong speliing")

"i don't done like is wrong spelling"

## Negation Handeling

In [45]:
import spacy

In [46]:
# Load the SpaCy English model
nlp = spacy.load("en_core_web_sm")

def handle_negation(text):
    """
    Add 'NOT_' prefix to words that are negated.
    """
    
    doc = nlp(text)
    negated = set()

    # Identify the heads of negation dependencies
    for token in doc:
        if token.dep_ == "neg":
            negated.add(token.head.i)

    # Apply NOT_ prefix to negated tokens
    result = []
    for i, token in enumerate(doc):
        if i in negated:
            result.append("NOT_" + token.text)
        else:
            result.append(token.text)

    return " ".join(result)

In [47]:
tweet1 = "I don't like the food."
tweet2 = "They never support the idea."
tweet3 = "This is not bad at all."

print(handle_negation(tweet1))  # I do not NOT_like the food .
print(handle_negation(tweet2))  # They never NOT_support the idea .
print(handle_negation(tweet3))  # This is not NOT_bad at all .


I do n't NOT_like the food .
They never NOT_support the idea .
This NOT_is not bad at all .


## Main Pipeline Function

In [48]:
def preprocessing_pipeline(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()  # Optional: Normalize casing & whitespace early

    text = smart_number_handler(text)              # Convert smart numbers to textual forms
    text = simplify_emoji_text(text)               # Convert emojis to meaningful words (e.g., 😊 → "smile")
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^\x00-\x7F]", ' ', text)     # Remove non-ASCII leftovers (e.g., fancy quotes, symbols)
    text = clean_tweet(text)                       # Remove mentions, hashtags, punctuation, excess spaces
    #text = handle_negation(text)                   # Tag negations BEFORE correction to preserve context ( remove negation handling as our transformers model can understand negations themselfs)
    text = replace_slangs(text)                    # Replace informal words (after spelling fix for max match)
    text = correct_text_spelling(text)             # Fix misspellings (after negations so “nott” becomes “not”)
    text = smart_number_handler(text)              # Run again if slang/spell fix introduces numbers

    return text

In [49]:
preprocessing_pipeline("I don't like the food thos is weong speliing 5/10 @pk ")

"i don't like the food this is wrong spelling rating"

In [50]:


tweets = X_train['Tweet'].astype(str).tolist()

# Use threads instead of processes
processed_tweets = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(tweets)
)

X_train['Cleaned_Tweet'] = processed_tweets

100%|██████████| 183861/183861 [01:32<00:00, 1998.37it/s]


## Lemmatization and Stop words

In [51]:
#  Download once, outside of the function
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [52]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [53]:
# Helper for POS
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default to noun

In [54]:
# Now safe for multiprocessing
def lemmatize_and_filter(text):
    if not isinstance(text, str):
        return ""

    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)

    cleaned_tokens = []
    for token, tag in tagged_tokens:
        token_lower = token.lower()
        if token_lower not in stop_words or token.startswith("NOT_"):
            lemma = lemmatizer.lemmatize(token_lower, get_wordnet_pos(tag))
            cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)


In [55]:
sample_text = "I do NOT_like eating apples because they are NOT_good and sometimes make me feel tired."

result = lemmatize_and_filter(sample_text)
print("Original:", sample_text)
print("Lemmatized & Stopword-filtered:", result)


Original: I do NOT_like eating apples because they are NOT_good and sometimes make me feel tired.
Lemmatized & Stopword-filtered: not_like eat apple not_good sometimes make feel tired .


In [56]:
# Convert column to list
texts = X_train['Cleaned_Tweet'].astype(str).tolist()

# Use threads for lightweight operations
lemmatized = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(texts)
)

# Store results
X_train['Final_Cleaned_Tweet'] = lemmatized

100%|██████████| 183861/183861 [03:55<00:00, 781.18it/s]


In [57]:
X_train.head().to_csv('sample.csv',index=False)

In [58]:
# ----------- For Test Set -------------
# Preprocessing
test_tweets = X_test['Tweet'].astype(str).tolist()
processed_test = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(test_tweets)
)
X_test['Cleaned_Tweet'] = processed_test

# Lemmatization
lemmatized_test = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(X_test['Cleaned_Tweet'].astype(str))
)
X_test['Final_Cleaned_Tweet'] = lemmatized_test

100%|██████████| 20000/20000 [00:25<00:00, 796.49it/s]


In [59]:
# ----------- For Validation Set -------------
# Preprocessing
val_tweets = X_val['Tweet'].astype(str).tolist()
processed_val = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(val_tweets)
)
X_val['Cleaned_Tweet'] = processed_val

# Lemmatization
lemmatized_val = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(X_val['Cleaned_Tweet'].astype(str))
)
X_val['Final_Cleaned_Tweet'] = lemmatized_val

100%|██████████| 15000/15000 [00:18<00:00, 806.76it/s]


In [60]:
X_val.head()

,Tweet,Cleaned_Tweet,Final_Cleaned_Tweet
0,Hey you stupid ass trolls that feel big going ...,hey you stupid ass trolls that feel big going ...,hey stupid as troll feel big go people donate ...
1,Ricky Schroder calls on American truckers to â...,ricky schroeder calls on american truckers to ...,ricky schroeder call american trucker shut was...
2,"In the latest episode of #FullComment, @Andrew...",in the latest episode of fullcomment joins to ...,late episode fullcomment join discuss freedomc...
3,"The erroneously self-christened ""Freedom"" Conv...",the erroneously self christened freedom convoy...,erroneously self christen freedom convoy fasci...
4,#FreedomConvoy2022 #FreedomConvoy #TruckersFor...,freedomconvoy2022 freedomconvoy truckersforfre...,freedomconvoy2022 freedomconvoy truckersforfre...


In [61]:
X_train.to_csv('X_train.csv',index=False)
X_test.to_csv('X_test.csv',index=False)
X_val.to_csv('X_val.csv',index=False)
y_train.to_csv('y_train.csv',index=False)
y_test.to_csv('y_test.csv',index=False)
y_val.to_csv('y_val.csv',index=False)

In [62]:
X_train.sample(5)

,Tweet,Cleaned_Tweet,Final_Cleaned_Tweet
80388,Kim Iversen: GoFundMe SEIZES Millions In Freed...,kim diverse gofundme seizes millions in freedo...,kim diverse gofundme seize million freedom con...
31886,This is how you fight fire with fire. \n\nHac...,this is how you fight fire with fire hackers l...,fight fire fire hacker leak name 'freedom conv...
58462,The remaining #OttawaOccupation\n are not prot...,the remaining ottawaoccupation are not protest...,remain ottawaoccupation protester 's cult dedi...
92144,"The U.S. Freedom Convoy, modeled after the Can...",the you a freedom convoy modelled after the ca...,freedom convoy model canadian trucker group pr...
144117,The Justice Centre is pleased to announce that...,the justice centre is pleased to announce that...,justice centre please announce crown stay char...
